In [ ]:
#libraries
import sys
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime as dt, timedelta


from keras.layers import LSTM
from keras.layers import Dense
from keras.models import Sequential

from pmdarima import auto_arima

from sklearn.preprocessing import MinMaxScaler

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from sklearn.metrics import max_error as maxe

from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_percentage_error as mape

#set warnings' mode
warnings.filterwarnings("ignore")

In [ ]:
#function columns' names
quantile_col = "quantile"
eq_comb_col = "equally_combined"
mse_comb_col = "mse_combined"

#date utils
def cast_year(d):
    return pd.DatetimeIndex(d).year

def cast_month(d):
    return pd.DatetimeIndex(d).month

#decimal utils
def format_decimal(s):
    return "{0:}".format(float(round(s, 0)))

#form quantile column
def get_quantile_col_name(quantile):
    return "weight_quantile_" + str(int(quantile*100))

def append_quantile_col(dfi, quantile, col):
    col_list = dfi.columns.to_list()
    col_list.append(col)
    dfi[date_col] = pd.to_datetime(dfi[date_col])
    week_col = dfi[date_col].dt.to_period("W").dt.to_timestamp()
    df_week = (dfi.groupby([week_col])[weight_col]
               .sum()).reset_index()
    if df_week[date_col][0].year != dfi[date_col][0]:
        df_week = df_week.iloc[1:, :]
    if df_week[date_col][df_week.shape[0]-1].year != dfi[date_col][df_week.shape[0]-1]:
        df_week = df_week.iloc[:-1, :]
    df_week[date_col] = pd.to_datetime(df_week[date_col], format = "%Y-%m")
    month_year_col = dfi[date_col].dt.strftime("%Y-%m")
    month_year_week_col = df_week[date_col].dt.strftime("%Y-%m")
    date_col_str = date_col + "_str"
    dfq = (df_week.groupby([month_year_week_col])[weight_col]
           .quantile(quantile)).reset_index(name = col)
    dfi[date_col_str] = month_year_col.astype(str)
    try:
        dfi = (dfi.merge(dfq, left_on=date_col_str, right_on=date_col)
               .rename(columns={date_col + "_x": date_col}))[col_list]
    except KeyError:
        pass
    dfq[col] = round(dfq[col], 3)
    dfi[col] = round(dfi[col], 3)
    return dfi, dfq

#get forecasts
def sma_forecast(df_train, col, col_name):
    sma = ARIMA(df_train[col], order=(0,0,1))
    sma_fit = sma.fit()
    sma_pred = sma_fit.predict(start=0, end=11)
    return get_forecast_df(sma_pred, col_name)

def holt_winters_forecast(df_train, col, col_name):
    hw = ExponentialSmoothing(df_train[col], trend="add",
                              damped_trend=False, 
                              seasonal="add", 
                              seasonal_periods=12)
    hw_fit = hw.fit()
    hw_pred = hw_fit.predict(start=0, end=11)
    return get_forecast_df(hw_pred, col_name)
    
def sarima_forecast(df_train, col, col_name):
    sarima = auto_arima(df_train[col], suppres_warnings=True,
                        seasonal=True, stepwise=True, m=12)
    sarima_pred = sarima.predict(n_periods=12)
    return get_forecast_df(sarima_pred, col_name)

def lstm_forecast(df_train, col, col_name, offset, lstm_model, scaler_model):
    df_scaled = scaler_model.fit_transform(df_train[[col]].values.reshape(-1, 1))
    
    train_x, train_y = [], []
    for ip in range(len(df_scaled)-offset-1):
        train_x.append(df_scaled[ip:(ip+offset), 0])
        train_y.append(df_scaled[ip + offset, 0])
    train_x, train_y = np.array(train_x), np.array(train_y)
    lstm_model.fit(train_x, train_y, epochs=120, batch_size=1, verbose=0)

    x_pred = train_x[-12:]
    y_pred = lstm_model.predict(x_pred)
    y_pred_scaled = scaler_model.inverse_transform(y_pred)
    return get_forecast_df(pd.DataFrame(data=y_pred_scaled)[0].astype("float64"), col_name)

#form forecast dataframe
def get_forecast_df(series, col_name):
    result = series.to_frame().reset_index(drop=True).round(3)
    result.columns = [col_name]
    return result

#break into train and test datasets
def split_train_test_df(dfq, test_year):
    train_split = dfq[cast_year(dfq[date_col]) < test_year]
    test_split = dfq[cast_year(dfq[date_col]) == test_year]
    return train_split, test_split

#calculate errors
ERRORS_LIST = ["mape", "mse", "maxe"]
def calculate_errors(q, m_list, d, ds, f):
    for m in m_list:
        d[q][m] = {}
        d[q][m][ERRORS_LIST[0]] = round(mape(ds[m].to_list(), ds[f].to_list()), 4)
        d[q][m][ERRORS_LIST[1]] = round(mse(ds[m].to_list(), ds[f].to_list()), 4)
        d[q][m][ERRORS_LIST[2]] = round(maxe(ds[m].to_list(), ds[f].to_list()), 4)
    
#define combination meta models
def combine_by_equal_prop(ds):
    coef = 1.0/(ds.shape[1] - 1)
    ds[eq_comb_col] = 0
    for col in ds.columns.to_list():
        if col != quantile_col:
            ds[eq_comb_col] += ds[col]*coef
    ds[eq_comb_col] = round(ds[eq_comb_col], 3)

def combine_by_mse_prop(ds, err_dict, c_dict):
    ds[mse_comb_col] = 0
    for qu, v in err_dict.items():
        err_list = [val["mse"] for val in list(v.values())]
        mse_sum = sum(err_list)
        mse_del_sum = sum([(1 - err/mse_sum) for err in err_list])
        
        c_dict[qu] = {}
        for me, er in v.items():
            coef = (1 - er["mse"]/mse_sum)/mse_del_sum
            c_dict[qu][me] =  coef
            for i in range(ds.shape[0]):
                if ds[quantile_col][i] == qu:
                    ds[mse_comb_col][i] += coef*ds[me][i]
    ds[mse_comb_col] = ds[mse_comb_col].round(3)

In [ ]:
#load dataset
df = pd.read_excel("cargo.xlsx", index_col=False, header=1).iloc[:, 1:]
df.transpose()

In [ ]:
#cast date column to datetime format
date_col = "onboard_date"
df[date_col] = pd.to_datetime(df[date_col],
                              format = "%Y-%m-%d",
                              errors='coerce')
df = df[df[date_col] != np.isnat(df[date_col])]
df = df.sort_values(date_col)

#drop NaN values
df = df.dropna()
df.reset_index(inplace=True)

#simplify dataset
weight_col = "weight"
cost_col = "customer_cost_with_tax"
per_col = "customer_cost_per_weight"
df = df[[date_col, weight_col, cost_col]]
df = (df.groupby([date_col])[[weight_col, cost_col]]
      .sum()).reset_index()
df[per_col] = round(df[cost_col] / df[weight_col], 2)
df = df[[date_col, weight_col, per_col]]

#save new dataset
df.to_csv("cargo_simple.csv", index=False)
df.head()

In [ ]:
#draw boxplot
df = pd.read_csv("cargo_simple.csv")
ax_box = df.boxplot(column=weight_col,
                    flierprops={"marker": "o",
                                "markersize": 4,
                                "markerfacecolor": "fuchsia"})

#drop outliers (IQR method)
Q3 = np.percentile(df[weight_col], 75, method="midpoint")
Q1 = np.percentile(df[weight_col], 25, method="midpoint")
IQR = Q3 - Q1

out_cnt = df[df[weight_col] > Q3+1.5*IQR].shape[0]
cnt = df.shape[0]
print(f"Outliers' ratio: {out_cnt} / {cnt} ≈ {round(out_cnt/cnt, 4)}")

df_clean = df[df[weight_col] < Q3+1.5*IQR]
df_clean.reset_index(inplace=False)

#draw boxplot
ax_box.figure.set_size_inches(6, 4)
mpl.rcParams.update({"font.size": 10})
plt.show()

In [ ]:
#calcaulate quantile values
QUANTILE_VALUE = 0.9
weight_quant_col = get_quantile_col_name(QUANTILE_VALUE)
df_clean, df_qun = append_quantile_col(df_clean, QUANTILE_VALUE, weight_quant_col)

#draw line plot 
x_dates = df_clean[date_col]
ax = plt.gca()
ax.plot(x_dates, df_clean[weight_col], linewidth=0.2,
        color="lightskyblue", label=weight_col)
ax.plot(x_dates, df_clean[weight_quant_col], linewidth=3,
        color="blue", label=f"{weight_col} [{int(QUANTILE_VALUE*100)}%-quantile]")

x = mdates.date2num(x_dates)
z = np.polyfit(x, df_clean[weight_quant_col], 1)
p = np.poly1d(z)
trend_equation = "y=%.4fx%.4f"%(z[0],z[1])
ax.plot(x, p(x), color="black", linewidth=2, linestyle="--",
        label=f"trend: {weight_col} [{int(QUANTILE_VALUE*100)}%-quantile]\n{trend_equation}")

ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1, 13, 1)))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.figure.set_size_inches(11, 3)
plt.setp(ax.get_xticklabels(), rotation=90)
mpl.rcParams.update({"font.size": 10})
plt.ylabel(weight_col, size=11)
plt.xlabel(date_col, size=11)
plt.legend(fontsize=8, loc="lower right")
plt.show()

In [ ]:
#seasonal decompose 
MODEL_OPTIONS_LIST = ["additive", "multiplicative"]
df_qun.set_index(date_col, inplace=True)

auto_corr_eval_opt = 2
model_option_opt = None
ax_sd = None
for model_option in MODEL_OPTIONS_LIST:
    sd = seasonal_decompose(df_qun, model=model_option, period=12)
    auto_corr_eval = durbin_watson(sd.resid.dropna())
    if abs(2 - auto_corr_eval) < auto_corr_eval_opt:
        auto_corr_eval_opt = auto_corr_eval
        model_option_opt = model_option
        ax_sd = sd.plot()

print(f"{model_option_opt}_model")
ax_sd.figure.set_size_inches(10, 3)
plt.xticks(rotation=90)
plt.show()

In [ ]:
#define models' names
sma_name = "simple_moving_average"
hw_name = "holt_winters"
sarima_name = "sarima"
lstm_name = "lstm"

In [ ]:
#build lstm model
scaler = MinMaxScaler(feature_range=(0,1))

lstm = Sequential()
lstm.add(LSTM(128, return_sequences=True,
              input_shape=(df_qun.shape[0]-12, 1)))
lstm.add(LSTM(64, return_sequences=False))
lstm.add(Dense(32))
lstm.add(Dense(1))
lstm.compile(optimizer="adam", loss="mean_squared_error")

In [ ]:
#build forecast plans for every model
fact_col = "fact"
OFFSET = 1
QUANTILES_LIST = [0.8, 0.85, 0.9, 0.9545, 0.9973]
TARGET_YEAR = 2021
METHODS_LIST = [sma_name, hw_name, sarima_name, lstm_name]

df_quan_dict = {}
errors_dict = {}
df_plan = pd.DataFrame()
for quan in QUANTILES_LIST:
    weight_quan_col = get_quantile_col_name(quan)
    _, df_quan = append_quantile_col(df_clean, quan, weight_quan_col)
    df_quan_dict[str(float(round(quan*100, 2)))] = df_quan
    train, test = split_train_test_df(df_quan, TARGET_YEAR)
    sma_plan = sma_forecast(train, weight_quan_col, sma_name)
    hw_plan = holt_winters_forecast(train, weight_quan_col, hw_name)
    sarima_plan = sarima_forecast(train, weight_quan_col, sarima_name)
    lstm_plan = lstm_forecast(train, weight_quan_col, lstm_name, OFFSET, lstm, scaler)
    
    quan_arr = np.empty(12)
    quan_key = str(round(quan*100, 2))
    quan_arr.fill(quan_key)
    quan_plan = pd.DataFrame(quan_arr)
    quan_plan.columns = [quantile_col]
    current_plan = pd.concat([quan_plan, sma_plan, hw_plan, 
                              sarima_plan, lstm_plan, 
                              get_forecast_df(test[weight_quan_col], fact_col)], 
                              axis=1)
    
    current_plan[quantile_col] = current_plan[quantile_col].astype("str")
    df_plan = pd.concat([df_plan, current_plan], axis=0)

    #calculate errors
    errors_dict[quan_key] = {}
    calculate_errors(quan_key, METHODS_LIST,
                     errors_dict, current_plan, fact_col)

df_plan.reset_index(inplace=True, drop=True)

In [ ]:
#combine forecasts
COMB_METHODS_LIST = [eq_comb_col, mse_comb_col]
mse_coef_map = {}

combine_by_equal_prop(df_plan)
combine_by_mse_prop(df_plan, errors_dict, mse_coef_map)

for qu in list(errors_dict.keys()):
    plan_cut = df_plan[df_plan[quantile_col] == qu]
    calculate_errors(qu, COMB_METHODS_LIST,
                     errors_dict, plan_cut, fact_col)

In [ ]:
method_col = "method"
quantiles_list = list(errors_dict.keys())
approaches_list = (METHODS_LIST + COMB_METHODS_LIST)
multi = pd.MultiIndex.from_arrays([list(np.repeat(quantiles_list, len(approaches_list))), 
                                  approaches_list*len(quantiles_list)], 
                                  names=[quantile_col, method_col])
err_df_list_data = []
for k1, v1 in errors_dict.items():
    for k2, v2 in v1.items():
        err_list = []
        for e in ERRORS_LIST:
            err_list.append(v2[e])
        err_df_list_data.append(err_list)
df_err = pd.DataFrame(data= err_df_list_data,
                      index=multi, 
                      columns=ERRORS_LIST)

targe_err_col = ERRORS_LIST[1]
df_best_quan_err = df_err.groupby(quantile_col)[targe_err_col].min()
df_err = df_err.merge(df_best_quan_err, left_index=True, right_index=True)
df_err = ((df_err[df_err[targe_err_col + "_x"] == df_err[targe_err_col + "_y"]]
          .rename({targe_err_col + "_x": targe_err_col}, axis=1))
          .drop(columns=targe_err_col + "_y"))
df_err = df_err.reset_index()

df_err_temp = df_err.copy()
df_err_temp[quantile_col] = round(df_err_temp[quantile_col].astype("float64")/100, 4).astype("str")

df_err_temp

In [ ]:
#model ideal scenario
left_col = "left_weight"
trucks_col = "free_trucks"
upcoming_col = "upcoming_trucks"
trucks_plan_col = "trucks"
income_col = "income"
mode_col = "mode"
mode_perfect = "\"perfect\""

SELF_TRUCKS_COUNT = 5 #Величина собственного автопарка
TRUCK_CAPACITY_TONE_COUNT = 14 #Грузоподъёмность т/с
TRIP_DAYS_COUNT = 4 #Длительность кругового маршрута
RENT_TRUCK_PRICE = 250000 #Стоимость аутсорса услуг водителя с т/с

df_part = df_clean[cast_year(df_clean[date_col]) == TARGET_YEAR+1]
df_part = df_part[[date_col, weight_col, per_col]]
df_part = df_part.reset_index(drop=True)
df_part.sort_values(by=[date_col, weight_col, per_col],
                    ascending=[True, True, False])
df_res_cols = [mode_col, date_col, trucks_plan_col, income_col]
df_res = pd.DataFrame(columns = df_res_cols)
rent_trucks_count = 0

for mon in set(cast_month(df_part[date_col]).to_list()):
    df_par = df_part[cast_month(df_part[date_col]) == mon].reset_index(drop=True)
    while True:
        failure_flag = False

        income = 0
        df_par[left_col] = 0
        df_par[trucks_col] = 0
        df_par[upcoming_col] = 0
        trucks_init = SELF_TRUCKS_COUNT + rent_trucks_count
        current_day = df_par[date_col][0]
        
        for day in range(df_par.shape[0]):
            if day == 0:
                trucks_avail_count = trucks_init
            else:
                trucks_avail_count = df_par[trucks_col][day-1]
            
            trucks_needed_count = math.ceil(df_par[weight_col][day]/TRUCK_CAPACITY_TONE_COUNT)
            trucks_left_count = trucks_avail_count + df_par[upcoming_col][day] - trucks_needed_count
            if trucks_left_count < 0:
                rent_trucks_count += 1
                failure_flag = True
                break
    
            current_day = df_par[date_col][day]
            df_par[trucks_col][day] = trucks_left_count
            income += df_par[weight_col][day]*df_par[per_col][day]
            upcoming_day = current_day + timedelta(days=TRIP_DAYS_COUNT+1)
            
            next_day_count = 0
            future_day = current_day
            while upcoming_day <= future_day:
                future_day = df_par[date_col][day+next_day_count]
                next_day_count += 1
                     
            df_par[upcoming_col][day+next_day_count] = trucks_needed_count
            
        if failure_flag is False:
            rent_trucks_month_count = SELF_TRUCKS_COUNT + rent_trucks_count
            df_res_data = [mode_perfect, dt.strptime(str(current_day).split(" ")[0][:-3], "%Y-%m"),
                           rent_trucks_month_count,
                           format_decimal(income - rent_trucks_count*RENT_TRUCK_PRICE)]
            df_res_cur = pd.DataFrame(data=[df_res_data],
                                      columns=df_res_cols)
            df_res = pd.concat([df_res, df_res_cur], axis = 0)
            rent_trucks_count = 0
            break
df_res = df_res.reset_index(drop=True)

In [ ]:
#fit best quantile forecast
df_fin_dict = {}
fore = None
fore_comb = None
for r in range(df_err.shape[0]):
    quan = df_err[quantile_col][r]
    train, test = split_train_test_df(df_quan_dict[quan], TARGET_YEAR+1)
    tar_col = train.columns[train.shape[1]-1]
    if df_err[method_col][r] == sma_name:
        fore = sma_forecast(train, tar_col, sma_name)
    if df_err[method_col][r] == hw_name:
        fore = holt_winters_forecast(train, tar_col, hw_name)
    if df_err[method_col][r] == sarima_name:
        fore = sarima_forecast(train, tar_col, sarima_name)
    if df_err[method_col][r] == lstm_name:
        fore = lstm_forecast(train, tar_col, lstm_name, OFFSET, lstm, scaler)
    if df_err[method_col][r] in [eq_comb_col, mse_comb_col]:
        fore1 = sma_forecast(train, tar_col, sma_name)
        fore2 = holt_winters_forecast(train, tar_col, hw_name)
        fore3 = sarima_forecast(train, tar_col, sarima_name)
        fore4 = lstm_forecast(train, tar_col, lstm_name, OFFSET, lstm, scaler)
        
    if df_err[method_col][r] == eq_comb_col:
        combine_by_equal_prop(fore_comb)
        fore = fore_comb[eq_comb_col]
    if df_err[method_col][r] == mse_comb_col:       
        fore = get_forecast_df((mse_coef_map[quan][sma_name]*fore1)[sma_name] + \
                               (mse_coef_map[quan][hw_name]*fore2)[hw_name] + \
                               (mse_coef_map[quan][sarima_name]*fore3)[sarima_name] + \
                               (mse_coef_map[quan][lstm_name]*fore4)[lstm_name], 
                               mse_comb_col)
    
    df_fin_dict[quan] = fore

In [ ]:
trucks_rent_col = "trucks_rented"
df_res_alt = pd.DataFrame(columns = df_res_cols)

for q, f in df_fin_dict.items():
    for mon in set(cast_month(df_part[date_col]).to_list()):
        df_par = df_part[cast_month(df_part[date_col]) == mon].reset_index()
        
        rent_trucks_count = math.ceil((f.iloc[mon-1, 0] - 
                       SELF_TRUCKS_COUNT*TRUCK_CAPACITY_TONE_COUNT)/TRUCK_CAPACITY_TONE_COUNT)
        if rent_trucks_count < 0:
            rent_trucks_count = 0
     
        income = 0
        df_par[left_col] = 0
        df_par[trucks_col] = 0
        df_par[upcoming_col] = 0
        trucks_init = SELF_TRUCKS_COUNT + rent_trucks_count
        current_day = df_par[date_col][0]
    
        for day in range(df_par.shape[0]):
            if day == 0:
                trucks_avail_count = trucks_init
            else:
                trucks_avail_count = df_par[trucks_col][day-1]
    
            trucks_needed_count = math.ceil(df_par[weight_col][day]/TRUCK_CAPACITY_TONE_COUNT)
            trucks_left_count = trucks_avail_count + df_par[upcoming_col][day] - trucks_needed_count
            if trucks_left_count < 0:
                continue
    
            current_day = df_par[date_col][day]
            df_par[trucks_col][day] = trucks_left_count
            income += df_par[weight_col][day]*df_par[per_col][day]
            upcoming_day = current_day + timedelta(days=TRIP_DAYS_COUNT+1)
    
            next_day_count = 0
            future_day = current_day
            while upcoming_day <= future_day:
                future_day = df_par[date_col][day+next_day_count]
                next_day_count += 1
    
            df_par[upcoming_col][day+next_day_count] = trucks_needed_count
    
    
        rent_trucks_month_count = SELF_TRUCKS_COUNT + rent_trucks_count
        df_res_data = [q, dt.strptime(str(current_day).split(" ")[0][:-3], "%Y-%m"),
                       rent_trucks_month_count,
                       format_decimal(income - rent_trucks_count*RENT_TRUCK_PRICE)]
        df_res_cur = pd.DataFrame(data=[df_res_data],
                                  columns=df_res_cols)
        df_res_alt = pd.concat([df_res_alt, df_res_cur], axis = 0)

df_res = pd.concat([df_res, df_res_alt], axis = 0)
df_res[trucks_rent_col] = df_res[trucks_plan_col] - SELF_TRUCKS_COUNT
df_res = df_res.reset_index(drop=True)

In [ ]:
#draw the results
currency_append = "(rub)"
qu_list = df_res[mode_col].to_list()
qu_plot = sorted(set(qu_list), key=qu_list.index)
target_dates = x_dates[x_dates.dt.year == TARGET_YEAR+1].dt.strftime("%Y-%m") \
               .drop_duplicates().reset_index(drop=True)

for qu in qu_plot:
    df_plot = df_res[df_res[mode_col] == qu]
    income_sum = df_plot[income_col].astype("float64").sum()
    fig, ax1 = plt.subplots()
    ax2 = ax1.twinx()
    
    ax1.bar(df_plot[date_col], df_plot[trucks_rent_col], 
            color="dimgray", edgecolor="black", width=8)
    ax2.plot(df_plot[date_col], df_plot[income_col].astype("float64"), 
             color="royalblue", marker="o", mec="black", linewidth=3)
    
    ax1.set_xlabel(date_col, fontsize=14)
    ax1.set_ylabel(trucks_rent_col, color="dimgray", fontsize=14)
    ax2.set_ylabel(f"{income_col} {currency_append}", color="royalblue", fontsize=14)

    qu_formatted = None
    if qu != mode_perfect:
        qu_formatted = str(round(float(qu)/100, 4))
        plt.title(f"quantile: {qu_formatted}, total_profit: {income_sum} {currency_append}\n", 
                  fontsize=14)
    else:
        plt.title(f"mode: {mode_perfect}, total_profit: {income_sum} {currency_append}\n", 
                  fontsize=14)
    
    ax2.yaxis.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1, 13, 1)))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax1.figure.set_size_inches(8, 2)
    plt.setp(ax1.get_xticklabels(), rotation=90)
    mpl.rcParams.update({"font.size": 11}) 
    plt.show()

In [ ]:
for qu in qu_plot:
    if qu != mode_perfect:
        ax3 = plt.gca()
        ax3.figure.set_size_inches(5, 2)
        for_col = pd.concat([target_dates, df_fin_dict[qu]], axis = 1)
        forecast_col = for_col.merge(df_part[date_col].dt.strftime("%Y-%m"), on=date_col)
        ax3.plot(df_part[date_col], forecast_col.iloc[:, 1], linewidth=3, color="lightblue")
        ax3.xaxis.set_major_locator(mdates.MonthLocator(bymonth=range(1, 13, 1)))
        ax3.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
        mpl.rcParams.update({"font.size": 8})
        plt.setp(ax3.get_xticklabels(), rotation=90)
        plt.ylabel(f"{quantile_col}_{weight_col}", size=8)
        plt.xlabel(date_col, size=8)
        plt.show()